# Fine-tune Qwen2-VL-7B for OMR on A100

Train `Qwen/Qwen2-VL-7B-Instruct` to decode sheet-music images into space-separated PrIMuS semantic strings.

| Item | Configuration |
|---|---|
| Dataset | `consolidated.zip` (~139k image/semantic pairs) |
| Method | BF16 LoRA on language attention/MLP, vision attention/MLP, and `lm_head` |
| Target GPU | NVIDIA A100 40 GB or 80 GB |
| Sampling | 50% handwritten, 25% clean, 25% camera; balanced original/corrupt within style |
| Default schedule | 2 epochs (~24-36 h on A100 40 GB); set `NUM_EPOCHS = 3` for a longer run |
| Output | `MyDrive/omr/qwen2-vl-7b-handnew/` |

Unlike Florence-2, Qwen uses its native tokenizer and chat template. Loss is masked so it applies only to the assistant's PrIMuS transcription—not the image or prompt. Native tokenization preserves Qwen's pretrained language model; the target remains the same PrIMuS text.

A100 optimizations: BF16, TF32, SDPA (Flash kernels selected automatically), fused AdamW, local-SSD data, persistent dataloader workers, **gradient checkpointing** (needed for 7B VRAM), and smaller VRAM-tier batch plans that keep effective batch size 32. Torch compile is intentionally disabled because variable image-token lengths cause expensive recompilation.

Before running:
1. Select an A100 runtime.
2. Upload `consolidated.zip` to `MyDrive/omr/`.
3. Upload `golden_set/` to `MyDrive/omr/golden_set/` for optional post-training evaluation.
4. Prefer a separate Colab session from any 2B training run so checkpoints stay isolated.


In [ ]:
# @title 1) Mount Drive and unpack training data to local SSD
from google.colab import drive

drive.mount("/content/drive")

import time
import zipfile
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/omr")
ZIP_PATH = DRIVE_ROOT / "consolidated.zip"
LOCAL_DATA_ROOT = Path("/content/omr_data")
CONSOLIDATED_DIR = LOCAL_DATA_ROOT / "consolidated"
IMAGES_DIR = CONSOLIDATED_DIR / "images"
GT_DIR = CONSOLIDATED_DIR / "ground_truth"

if IMAGES_DIR.is_dir() and any(IMAGES_DIR.iterdir()):
    print(f"Local data already present: {IMAGES_DIR}")
elif not ZIP_PATH.is_file():
    raise FileNotFoundError(f"Upload consolidated.zip to {ZIP_PATH}")
else:
    LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {ZIP_PATH} -> {LOCAL_DATA_ROOT} ...")
    started = time.time()
    with zipfile.ZipFile(ZIP_PATH, "r") as archive:
        archive.extractall(LOCAL_DATA_ROOT)
    print(f"Extracted in {(time.time() - started) / 60:.1f} minutes")

if not IMAGES_DIR.is_dir() or not GT_DIR.is_dir():
    raise FileNotFoundError("Expected consolidated/images and consolidated/ground_truth after extraction")
print(f"images={IMAGES_DIR}  ground_truth={GT_DIR}")

In [ ]:
# @title 2) Install Qwen2-VL training dependencies (restarts once)
# Pillow 12.0.0 can leave Colab with mixed PIL modules, while Colab's old
# optional torchao can fail PEFT's version gate. Repair both, then restart
# before importing PIL, transformers, or peft.
import importlib.machinery
import importlib.metadata
import os
import signal
import subprocess
import sys
from pathlib import Path

PILLOW_VERSION = "12.3.0"
_SETUP_REVISION = 2
_SETUP_MARKER = Path(
    f"/content/.qwen2_vl_7b_setup_v{_SETUP_REVISION}_pillow_"
    f"{PILLOW_VERSION.replace('.', '_')}_no_torchao_ready"
)


def _run_pip(*args: str) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", *args])


def _installed_version(distribution: str) -> str | None:
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None


def _torchao_files_importable() -> bool:
    # PathFinder checks files on sys.path without trusting a stale sys.modules entry.
    return importlib.machinery.PathFinder.find_spec("torchao") is not None


def _setup_is_valid() -> bool:
    return (
        _installed_version("pillow") == PILLOW_VERSION
        and _installed_version("torchao") is None
        and not _torchao_files_importable()
    )


# A revisioned marker makes runtimes created by the older Pillow-only setup
# perform this migration once instead of skipping the required torchao removal.
if _SETUP_MARKER.exists() and not _setup_is_valid():
    print("Setup marker is stale; repairing the environment ...", flush=True)
    _SETUP_MARKER.unlink()

if not _SETUP_MARKER.exists():
    print(f"Installing dependencies and Pillow {PILLOW_VERSION} ...", flush=True)
    _run_pip(
        "install",
        "-q",
        "--no-cache-dir",
        "-U",
        "transformers>=4.49.0,<5.0.0",
        "accelerate",
        "peft",
        "datasets",
        "qwen-vl-utils",
        "tqdm",
        "sentencepiece",
        "safetensors",
    )
    # Install Pillow last so dependency resolution cannot put 12.0.0 back.
    _run_pip(
        "install",
        "-q",
        "--no-cache-dir",
        "--force-reinstall",
        "--no-deps",
        f"pillow=={PILLOW_VERSION}",
    )
    # torchao is optional for this ordinary BF16 LoRA run. Removing Colab's
    # incompatible preinstall is safer than changing the PyTorch stack.
    _run_pip("uninstall", "-q", "-y", "torchao")

    if not _setup_is_valid():
        raise RuntimeError(
            "Dependency repair did not produce Pillow "
            f"{PILLOW_VERSION} with torchao absent. "
            f"pillow={_installed_version('pillow')}, "
            f"torchao={_installed_version('torchao')}, "
            f"torchao_importable={_torchao_files_importable()}"
        )
    _SETUP_MARKER.write_text(
        f"revision={_SETUP_REVISION}\npillow={PILLOW_VERSION}\ntorchao=absent\n",
        encoding="utf-8",
    )
    print(
        "Dependencies installed and incompatible torchao removed. Colab will restart now; "
        "reconnect, then rerun from cell 1. The setup marker prevents another restart.",
        flush=True,
    )
    os.kill(os.getpid(), signal.SIGKILL)
    raise RuntimeError("Automatic restart failed. Use Runtime → Restart session, then rerun from cell 1.")

if not _setup_is_valid():
    _SETUP_MARKER.unlink(missing_ok=True)
    raise RuntimeError(
        "The restarted environment is inconsistent. Rerun this cell to repair it and restart once. "
        f"pillow={_installed_version('pillow')}, "
        f"torchao={_installed_version('torchao')}, "
        f"torchao_importable={_torchao_files_importable()}"
    )

# These imports verify that the restarted process has one consistent Pillow
# install and that package metadata/import discovery both see torchao as absent.
import PIL
from PIL import ImageDraw, ImageText
from PIL._typing import _Ink

import accelerate
import peft
import torch
import transformers

print(f"torch={torch.__version__}")
print(f"transformers={transformers.__version__}")
print(f"peft={peft.__version__}")
print(f"accelerate={accelerate.__version__}")
print(f"pillow={PIL.__version__}")
print("torchao=absent (not needed for BF16 LoRA)")
print("Pillow ImageDraw/ImageText imports: OK")

In [ ]:
# @title 3) A100 configuration
from __future__ import annotations

import json
import os
import random
from pathlib import Path

import torch

MODEL_ID = "Qwen/Qwen2-VL-7B-Instruct"
DRIVE_OUTPUT_DIR = DRIVE_ROOT / "qwen2-vl-7b-handnew"
CHECKPOINT_DIR = DRIVE_OUTPUT_DIR / "checkpoints"
MANIFEST_PATH = DRIVE_ROOT / "qwen2-vl-7b-manifest.json"  # shared with smoke test
GOLDEN_ROOT = DRIVE_ROOT / "golden_set"

SYSTEM_PROMPT = "You are an optical music recognition system. Return only PrIMuS semantic tokens."
USER_PROMPT = (
    "Transcribe this monophonic sheet-music image into PrIMuS semantic format. "
    "Return only one space-separated token string with no explanation or markdown."
)

SEED = 1337
NUM_EPOCHS = 2  # ~24-36h on A100 40GB; set 3 only if you can resume across sessions
MAX_TRAIN_SAMPLES = None      # full dataset; smoke notebook overrides this
MAX_TRAIN_STEPS = -1          # epoch-based training; smoke notebook overrides this
# Intentional 7B LoRA recipe:
# - HF's Qwen2-VL-7B QLoRA example uses r=8 on q/v only at LR=2e-4.
# - We adapt every language attention/MLP projection, vision qkv/MLP, and lm_head,
#   so r=32 already provides substantially more adapter capacity. The broader target
#   set also warrants the more conservative LR=1e-4. Do not scale rank just because
#   the frozen base is larger; the successful 2B run used this same recipe.
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01
MAX_SEQ_LENGTH = 1536
MIN_PIXELS = 64 * 28 * 28
MAX_PIXELS = 768 * 768
LORA_R = 32
LORA_ALPHA = 64              # alpha/r = 2.0, matching the proven 2B scaling
LORA_DROPOUT = 0.05
SAVE_STEPS = 500
LOGGING_STEPS = 25
SAVE_TOTAL_LIMIT = 6
RESUME_FROM_CHECKPOINT = True
GOLDEN_EVAL_LIMIT = None      # all available manifest rows
EXPORT_MERGED_MODEL = False  # merging 7B needs lots of RAM; keep adapter-only by default
STYLE_WEIGHTS = {"handwritten": 0.50, "clean": 0.25, "camera": 0.25}

if not torch.cuda.is_available():
    raise RuntimeError("This notebook requires an NVIDIA A100 runtime.")
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
if "a100" not in GPU_NAME.lower():
    raise RuntimeError(f"Expected an A100, found {GPU_NAME}")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("A100 BF16 support was not detected")

# Qwen2-VL has variable visual sequence lengths, so static torch.compile graphs tend
# to recompile. SDPA still dispatches to A100 flash kernels where shapes permit.
# 7B needs a much smaller microbatch than 2B; keep effective batch = 32.
if VRAM_GB >= 70:
    PER_DEVICE_BATCH_SIZE, GRAD_ACCUM_STEPS = 2, 16
elif VRAM_GB >= 38:
    PER_DEVICE_BATCH_SIZE, GRAD_ACCUM_STEPS = 1, 32
else:
    raise RuntimeError(f"Only {VRAM_GB:.1f} GB VRAM is visible")

CPU_COUNT = os.cpu_count() or 4
DATALOADER_WORKERS = min(8, CPU_COUNT)  # slightly fewer workers to leave RAM for 7B
GRADIENT_CHECKPOINTING = True  # required for 7B LoRA on A100 40GB
TORCH_COMPILE = False
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")  # dataloader workers provide parallelism
torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(f"GPU: {GPU_NAME} ({VRAM_GB:.1f} GB)")
print(
    f"batch={PER_DEVICE_BATCH_SIZE} x accum={GRAD_ACCUM_STEPS} "
    f"(effective {PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS}) | "
    f"workers={DATALOADER_WORKERS} | bf16 + TF32 + SDPA | grad_ckpt={GRADIENT_CHECKPOINTING}"
)
print(f"LoRA: r={LORA_R}, alpha={LORA_ALPHA}, scaling={LORA_ALPHA / LORA_R:g}; LR={LEARNING_RATE:g}")
print(f"Output: {DRIVE_OUTPUT_DIR}")


In [ ]:
# @title 4) Build/load leak-free manifest and style-balanced sample weights
from collections import Counter, defaultdict

from tqdm.auto import tqdm

IMAGE_EXTS = {".png", ".jpg", ".jpeg"}
MANIFEST_VERSION = 1


def canonical_id(stem: str) -> str:
    return stem.split("__", 1)[1] if "__" in stem else stem


def classify_record(stem: str) -> tuple[str, str]:
    prefix = stem.split("__", 1)[0].lower()
    source = "corrupt" if prefix.startswith("corrupt_") else "original"
    if "handwritten" in prefix or prefix == "mashcima":
        style = "handwritten"
    elif "camera" in prefix:
        style = "camera"
    else:
        style = "clean"
    return style, source


def load_golden_ids() -> set[str]:
    manifest_path = GOLDEN_ROOT / "manifest.jsonl"
    ids: set[str] = set()
    if not manifest_path.is_file():
        print("Golden manifest not found; no ID-based leakage exclusions applied")
        return ids
    for line in manifest_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        for key in ("id", "incipit_id"):
            if row.get(key):
                ids.add(str(row[key]))
    print(f"Golden exclusion IDs: {len(ids):,}")
    return ids


def build_manifest() -> list[dict]:
    golden_ids = load_golden_ids()
    records: list[dict] = []
    image_paths = sorted(
        path for path in IMAGES_DIR.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTS
    )
    missing_gt = 0
    excluded = 0
    for image_path in tqdm(image_paths, desc="Scanning consolidated pairs"):
        gt_path = GT_DIR / f"{image_path.stem}.semantic"
        if not gt_path.is_file():
            missing_gt += 1
            continue
        if canonical_id(image_path.stem) in golden_ids:
            excluded += 1
            continue
        target = " ".join(gt_path.read_text(encoding="utf-8").split())
        if not target:
            continue
        style, source = classify_record(image_path.stem)
        records.append(
            {
                "image": str(image_path),
                "text": target,
                "stem": image_path.stem,
                "style": style,
                "source": source,
            }
        )
    print(f"pairs={len(records):,} excluded={excluded:,} missing_gt={missing_gt:,}")
    return records


def load_or_build_manifest() -> list[dict]:
    if MANIFEST_PATH.is_file():
        payload = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
        if payload.get("version") == MANIFEST_VERSION:
            print(f"Loaded cached manifest: {len(payload['records']):,} pairs")
            return payload["records"]
    records = build_manifest()
    MANIFEST_PATH.write_text(
        json.dumps({"version": MANIFEST_VERSION, "records": records}),
        encoding="utf-8",
    )
    print(f"Cached manifest -> {MANIFEST_PATH}")
    return records


def stratified_limit(records: list[dict], limit: int | None) -> list[dict]:
    if not limit or limit >= len(records):
        return records
    rng = random.Random(SEED)
    buckets: dict[tuple[str, str], list[dict]] = defaultdict(list)
    for record in records:
        buckets[(record["style"], record["source"])].append(record)
    selected: list[dict] = []
    for style, style_fraction in STYLE_WEIGHTS.items():
        available_sources = [source for source in ("original", "corrupt") if buckets[(style, source)]]
        if not available_sources:
            continue
        style_quota = round(limit * style_fraction)
        source_quota = style_quota // len(available_sources)
        for source in available_sources:
            bucket = buckets[(style, source)]
            selected.extend(rng.sample(bucket, min(source_quota, len(bucket))))
    if len(selected) < limit:
        selected_stems = {record["stem"] for record in selected}
        remainder = [record for record in records if record["stem"] not in selected_stems]
        selected.extend(rng.sample(remainder, min(limit - len(selected), len(remainder))))
    rng.shuffle(selected)
    return selected[:limit]


def build_sample_weights(records: list[dict]) -> list[float]:
    counts = Counter((record["style"], record["source"]) for record in records)
    weights: list[float] = []
    for record in records:
        style = record["style"]
        present_sources = [source for source in ("original", "corrupt") if counts[(style, source)] > 0]
        target_mass = STYLE_WEIGHTS[style] / len(present_sources)
        weights.append(target_mass / counts[(style, record["source"])])
    return weights

all_records = load_or_build_manifest()
train_records = stratified_limit(all_records, MAX_TRAIN_SAMPLES)
train_sample_weights = build_sample_weights(train_records)
print(f"Training records: {len(train_records):,}")
print("Strata:", dict(sorted(Counter((r['style'], r['source']) for r in train_records).items())))

In [ ]:
# @title 5) Load Qwen2-VL-7B and attach LoRA
import importlib.machinery
import importlib.metadata

from packaging.version import InvalidVersion, Version

TORCHAO_MIN_VERSION = Version("0.16.0")
try:
    torchao_version_text = importlib.metadata.version("torchao")
except importlib.metadata.PackageNotFoundError:
    torchao_version_text = None

torchao_importable = importlib.machinery.PathFinder.find_spec("torchao") is not None
if torchao_version_text is None:
    if torchao_importable:
        raise RuntimeError(
            "torchao files are importable but package metadata is missing. "
            "Rerun setup cell 2, allow the restart, then rerun from cell 1."
        )
    print("torchao preflight: absent (expected for BF16 LoRA)")
else:
    try:
        torchao_version = Version(torchao_version_text)
    except InvalidVersion as exc:
        raise RuntimeError(
            f"Cannot validate installed torchao version {torchao_version_text!r}. "
            "Rerun setup cell 2, allow the restart, then rerun from cell 1."
        ) from exc
    if torchao_version < TORCHAO_MIN_VERSION:
        raise RuntimeError(
            f"Incompatible torchao {torchao_version}; PEFT requires >= {TORCHAO_MIN_VERSION}. "
            "This notebook does not need torchao: rerun setup cell 2, allow the restart, "
            "then rerun from cell 1."
        )
    if not torchao_importable:
        raise RuntimeError(
            f"torchao {torchao_version} has metadata but cannot be imported. "
            "Rerun setup cell 2, allow the restart, then rerun from cell 1."
        )
    print(f"torchao preflight: compatible ({torchao_version})")

from peft import LoraConfig, get_peft_model
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

print(f"Loading {MODEL_ID} in BF16 with SDPA...")
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)
base_model.config.use_cache = False

# q/k/v/o + gated MLP target the language model; qkv/fc1/fc2 target Qwen's
# visual transformer; lm_head learns the exact structured output distribution.
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
        "qkv", "fc1", "fc2", "lm_head",
    ],
)
model = get_peft_model(base_model, lora_config)
model.config.use_cache = False
if GRADIENT_CHECKPOINTING:
    # PEFT + gradient checkpointing needs input grads on frozen base inputs.
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()
    else:
        base_model.enable_input_require_grads()
model.print_trainable_parameters()

vision_lora = [name for name, _ in model.named_parameters() if "visual" in name and "lora_" in name]
language_lora = [name for name, _ in model.named_parameters() if "model.layers" in name and "lora_" in name]
print(f"Vision LoRA tensors: {len(vision_lora)}")
print(f"Language LoRA tensors: {len(language_lora)}")
if not vision_lora:
    raise RuntimeError("No vision LoRA modules matched; inspect Qwen module names before training")


In [ ]:
# @title 6) Dataset and target-only multimodal collator
from PIL import Image
from torch.utils.data import Dataset

processor.tokenizer.padding_side = "right"
ASSISTANT_PREFIX_IDS = processor.tokenizer.encode(
    "<|im_start|>assistant\n",
    add_special_tokens=False,
)
if not ASSISTANT_PREFIX_IDS:
    raise RuntimeError("Could not encode Qwen assistant prefix")


class OMRDataset(Dataset):
    def __init__(self, records: list[dict]):
        self.records = records

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> dict:
        record = self.records[index]
        try:
            with Image.open(record["image"]) as source:
                image = source.convert("RGB")
                image.load()
        except (OSError, SyntaxError, ValueError) as exc:
            # Preserve batch shape if a bad file slips through; report it and use the next item.
            print(f"Unreadable image {record['image']}: {exc}")
            return self[(index + 1) % len(self.records)]
        return {"image": image, "text": record["text"], "stem": record["stem"]}


def find_last_subsequence(sequence: list[int], pattern: list[int]) -> int:
    for start in range(len(sequence) - len(pattern), -1, -1):
        if sequence[start:start + len(pattern)] == pattern:
            return start
    return -1


def make_messages(target: str | None = None) -> list[dict]:
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": USER_PROMPT},
            ],
        },
    ]
    if target is not None:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": target}]})
    return messages


class QwenOMRCollator:
    def __call__(self, examples: list[dict]) -> dict[str, torch.Tensor]:
        texts = [
            processor.apply_chat_template(
                make_messages(example["text"]),
                tokenize=False,
                add_generation_prompt=False,
            )
            for example in examples
        ]
        images = [example["image"] for example in examples]
        batch = processor(
            text=texts,
            images=images,
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
            return_tensors="pt",
        )

        labels = batch["input_ids"].clone()
        for row in range(labels.shape[0]):
            token_ids = batch["input_ids"][row].tolist()
            marker_start = find_last_subsequence(token_ids, ASSISTANT_PREFIX_IDS)
            if marker_start < 0:
                raise RuntimeError("Assistant marker missing after chat-template tokenization")
            target_start = marker_start + len(ASSISTANT_PREFIX_IDS)
            labels[row, :target_start] = -100
            labels[row, batch["attention_mask"][row] == 0] = -100
            if not torch.any(labels[row] != -100):
                raise RuntimeError(
                    f"Target was fully truncated; raise MAX_SEQ_LENGTH above {MAX_SEQ_LENGTH}"
                )
        batch["labels"] = labels
        return dict(batch)


train_dataset = OMRDataset(train_records)
data_collator = QwenOMRCollator()

# One-batch contract check before committing GPU hours.
probe = data_collator([train_dataset[0], train_dataset[1]])
print({key: tuple(value.shape) for key, value in probe.items() if hasattr(value, "shape")})
print("Supervised tokens:", int((probe["labels"] != -100).sum()))
del probe

In [ ]:
# @title 7) Train with style-balanced sampling and resume-safe Drive checkpoints
import glob

from torch.utils.data import WeightedRandomSampler
from transformers import Trainer, TrainingArguments


class StyleBalancedTrainer(Trainer):
    def __init__(self, *args, train_sample_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self._train_sample_weights = train_sample_weights

    def _get_train_sampler(self, train_dataset=None):
        dataset = self.train_dataset if train_dataset is None else train_dataset
        if self._train_sample_weights is not None and dataset is self.train_dataset:
            return WeightedRandomSampler(
                self._train_sample_weights,
                num_samples=len(dataset),
                replacement=True,
            )
        try:
            return super()._get_train_sampler(train_dataset)
        except TypeError:
            return super()._get_train_sampler()


def latest_checkpoint() -> str | None:
    checkpoints = glob.glob(str(CHECKPOINT_DIR / "checkpoint-*"))
    if not checkpoints:
        return None
    return max(checkpoints, key=lambda path: int(path.rsplit("-", 1)[-1]))


resume_path = latest_checkpoint() if RESUME_FROM_CHECKPOINT else None
if resume_path:
    print(f"Resuming from {resume_path}")

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=NUM_EPOCHS,
    max_steps=MAX_TRAIN_STEPS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=1.0,
    bf16=True,
    fp16=False,
    tf32=True,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch_fused",
    torch_compile=TORCH_COMPILE,
    logging_steps=LOGGING_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    eval_strategy="no",
    remove_unused_columns=False,
    label_names=["labels"],
    dataloader_drop_last=True,
    dataloader_num_workers=DATALOADER_WORKERS,
    dataloader_pin_memory=True,
    dataloader_persistent_workers=DATALOADER_WORKERS > 0,
    dataloader_prefetch_factor=4 if DATALOADER_WORKERS > 0 else None,
    report_to=[],
    seed=SEED,
    data_seed=SEED,
)

trainer = StyleBalancedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    train_sample_weights=train_sample_weights,
)
train_result = trainer.train(resume_from_checkpoint=resume_path)
print(train_result)

adapter_dir = DRIVE_OUTPUT_DIR / "final-adapter"
trainer.save_model(str(adapter_dir))
processor.save_pretrained(str(adapter_dir))
print(f"Saved LoRA adapter + processor -> {adapter_dir}")

In [ ]:
# @title 8) Evaluate the fine-tuned adapter on the golden manifest
import re
from collections import defaultdict

GROUP_DIRS = {
    "handwritten_original": ("original", "images", "handwritten"),
    "clean_original": ("original", "images", "clean"),
    "camera_distorted_original": ("original", "images", "camera"),
    "handwritten_corrupt": ("corrupt", "images", "handwritten"),
    "clean_corrupt": ("corrupt", "images", "clean"),
    "camera_distorted_corrupt": ("corrupt", "images", "camera"),
}


def levenshtein(left: list[str], right: list[str]) -> int:
    # Relative major/minor share the same accidentals; canonicalize to the major form.
    _RELATIVE_MINOR_TO_MAJOR = {
        "Am": "CM",
        "Em": "GM",
        "Bm": "DM",
        "F#m": "AM",
        "C#m": "EM",
        "G#m": "BM",
        "D#m": "F#M",
        "A#m": "C#M",
        "Dm": "FM",
        "Gm": "BbM",
        "Cm": "EbM",
        "Fm": "AbM",
        "Bbm": "DbM",
        "Ebm": "GbM",
        "Abm": "CbM",
    }
    _TIME_SIGNATURE_CANON = {
        "timeSignature-4/4": "timeSignature-C",
        "timeSignature-C": "timeSignature-C",
        "timeSignature-2/2": "timeSignature-C/",
        "timeSignature-C/": "timeSignature-C/",
    }

    def canonicalize(token: str) -> str:
        if token.startswith("keySignature-"):
            key = token[len("keySignature-") :]
            key = _RELATIVE_MINOR_TO_MAJOR.get(key, key)
            return f"keySignature-{key}"
        return _TIME_SIGNATURE_CANON.get(token, token)

    left = [canonicalize(t) for t in left]
    right = [canonicalize(t) for t in right]
    previous = list(range(len(right) + 1))
    for i, left_token in enumerate(left, 1):
        current = [i]
        for j, right_token in enumerate(right, 1):
            current.append(
                min(
                    current[-1] + 1,
                    previous[j] + 1,
                    previous[j - 1] + (left_token != right_token),
                )
            )
        previous = current
    return previous[-1]


def resolve_golden_image(row: dict) -> Path | None:
    group = row["group"]
    if group not in GROUP_DIRS:
        return None
    filename = str(row["image_path"]).replace("\\", "/").rsplit("/", 1)[-1]
    folder = GOLDEN_ROOT.joinpath(*GROUP_DIRS[group])
    exact = folder / filename
    if exact.is_file():
        return exact
    stem = Path(filename).stem
    return next((path for path in folder.glob(f"{stem}.*") if path.suffix.lower() in IMAGE_EXTS), None)


def load_golden_records() -> list[dict]:
    manifest_path = GOLDEN_ROOT / "manifest.jsonl"
    if not manifest_path.is_file():
        print(f"Skipping golden eval: {manifest_path} not found")
        return []
    records = []
    for line in manifest_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        image_path = resolve_golden_image(row)
        source = row.get("source", "original")
        gt_path = GOLDEN_ROOT / source / "ground_truth" / f"{row['id']}.semantic"
        if image_path and gt_path.is_file():
            records.append(
                {
                    "id": row["id"],
                    "group": row["group"],
                    "image": str(image_path),
                    "target": " ".join(gt_path.read_text(encoding="utf-8").split()),
                }
            )
    if GOLDEN_EVAL_LIMIT and len(records) > GOLDEN_EVAL_LIMIT:
        buckets: dict[str, list[dict]] = defaultdict(list)
        for record in records:
            buckets[record["group"]].append(record)
        limited = []
        while len(limited) < GOLDEN_EVAL_LIMIT and any(buckets.values()):
            for group in sorted(buckets):
                if buckets[group] and len(limited) < GOLDEN_EVAL_LIMIT:
                    limited.append(buckets[group].pop(0))
        records = limited
    return records


def clean_prediction(text: str) -> str:
    text = re.sub(r"```(?:\w+)?", "", text or "").replace("```", "")
    return " ".join(text.split())


def predict_one(image_path: str) -> str:
    with Image.open(image_path) as source:
        image = source.convert("RGB")
    prompt_text = processor.apply_chat_template(
        make_messages(),
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = processor(
        text=[prompt_text],
        images=[image],
        padding=True,
        return_tensors="pt",
    ).to(model.device)
    prompt_length = inputs["input_ids"].shape[1]
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            num_beams=1,
            use_cache=True,
        )
    generated = output_ids[:, prompt_length:]
    return clean_prediction(
        processor.tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
    )


golden_records = load_golden_records()
if golden_records:
    model.eval()
    rows = []
    for record in tqdm(golden_records, desc="Golden eval"):
        prediction = predict_one(record["image"])
        target_tokens = record["target"].split()
        prediction_tokens = prediction.split()
        edits = levenshtein(target_tokens, prediction_tokens)
        accuracy = (1.0 - edits / max(1, len(target_tokens))) * 100.0
        rows.append({**record, "prediction": prediction, "edits": edits, "accuracy_pct": accuracy})
    by_group = defaultdict(list)
    for row in rows:
        by_group[row["group"]].append(row["accuracy_pct"])
    summary = {
        "model_id": MODEL_ID,
        "n": len(rows),
        "mean_accuracy_pct": sum(row["accuracy_pct"] for row in rows) / len(rows),
        "exact_matches": sum(row["edits"] == 0 for row in rows),
        "by_group": {group: sum(values) / len(values) for group, values in sorted(by_group.items())},
        "samples": rows,
    }
    eval_path = DRIVE_OUTPUT_DIR / "eval_golden_after.json"
    eval_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(json.dumps({key: value for key, value in summary.items() if key != "samples"}, indent=2))
    print(f"Saved -> {eval_path}")

In [ ]:
# @title 9) Optionally merge LoRA and export standalone model
if EXPORT_MERGED_MODEL:
    merged_dir = DRIVE_OUTPUT_DIR / "final-merged"
    print("Merging LoRA into Qwen2-VL-7B base weights...")
    merged_model = trainer.model.merge_and_unload()
    merged_model.config.use_cache = True
    merged_model.save_pretrained(
        str(merged_dir),
        safe_serialization=True,
        max_shard_size="4GB",
    )
    processor.save_pretrained(str(merged_dir))
    print(f"Standalone merged model -> {merged_dir}")
else:
    print(f"Merged export disabled; adapter is at {DRIVE_OUTPUT_DIR / 'final-adapter'}")

print("Done. Resume checkpoints remain under:", CHECKPOINT_DIR)

## Multi-session resume

A full Qwen2-VL-7B run is expected to take roughly **24-36 hours for 2 epochs** on A100 40 GB (less on 80 GB). Drive checkpoints protect against runtime interruption.

1. Ensure checkpoints are on Drive: `MyDrive/omr/qwen2-vl-7b-handnew/checkpoints/checkpoint-*`
2. Re-open this notebook, run setup cells, then training with `RESUME_FROM_CHECKPOINT = True`
3. The Trainer restores model, optimizer, scheduler, and step state from the latest checkpoint
4. Mid-run golden evals: open `notebooks/eval_qwen2_vl_7b_latest_checkpoint_colab.ipynb` in a separate Colab session

To **force a fresh run**, delete `checkpoints/` on Drive (only if you changed split settings or want a clean start).
